# اليوم 4 — Kafka وجودة البيانات — Labs 05–06

أمل خوتاني — Amal Khotani.

ضمن **هندسة البيانات الحديثة لأنظمة الذكاء الاصطناعي — Modern Data Engineering for AI Systems (SDA-DSC-214)** لدى [أكاديمية سدايا](https://github.com/SDAIAAcademy). #SDAIAAcademy

مواد الدورة ودوالها: **ميعاد المري — Meaad Al-Marri**. البيانات اصطناعية.

هذه نسخة منظمة من قسم اليوم 4 في الدفتر المرفوع `amal_khotani (3).ipynb`؛ حُفظت شيفرة الخلايا الأصلية ومخرجاتها وأرقام تنفيذها كما وردت. لم يُعد تشغيل اللابات أثناء فصل الدفاتر، ولم يُختبر Run all في جلسة نظيفة. أرقام التنفيذ تعود إلى جلسات مختلفة.

قبل إعادة التشغيل، اتبع `docs/SETUP.md` في المستودع: Python 3.11، Java 17، PySpark 3.5.8، Delta Spark 3.3.3 وPy4J 0.10.9.9. يجب أن يكون مستودع المقرر وبياناته ومساحة العمل المطلوبة متاحين؛ وجود المخرجات المحفوظة وحده لا يجهز بيئة التشغيل.

ابدأ بعد إكمال اليوم 3؛ في جلسة جديدة استعد أرشيف ذلك اليوم إلى جذر المستودع وأعد تهيئة البيئة، ثم شغّل خلية Continue your project.


# **lab 4 **

 Continue your project

In [38]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))

Continue workspace: outputs/day01_bronze_xgnqdyt3


Receive Kafka events

In [47]:
import sys
import subprocess
import pathlib
import tarfile
import hashlib
import re
import os
import time

subprocess.run(
    [sys.executable, "-m", "pip", "install", "kafka-python==2.2.15"],
    check=True
)

from masar.streaming import stream_preflight
from kafka.admin import KafkaAdminClient

kafka_dir = pathlib.Path("/content/kafka_2.13-4.0.2")
kafka_state = pathlib.Path("/content/masar_kafka")
kafka_state.mkdir(exist_ok=True)
kafka_log = kafka_state / "server.log"

if not stream_preflight()["tcp_listening"]:
    if not (kafka_dir / "bin/kafka-server-start.sh").is_file():
        print("Downloading Kafka — maximum 2 minutes...", flush=True)

        url = "https://mirrors.huaweicloud.com/apache/kafka/4.0.2/kafka_2.13-4.0.2.tgz"
        package = pathlib.Path("/content/kafka-mirror.tgz")

        subprocess.run(
            [
                "curl", "-fLsS",
                "--connect-timeout", "15",
                "--max-time", "120",
                url, "-o", str(package)
            ],
            check=True,
            timeout=130
        )

        expected = (
            "b854b5ee761f04f8bd27f18576590cd54f4890d5dc103818f3db23c1a5b04de2"
            "7f8465d4042f7cdf463dbfa6f149dafca92b60f56b5e2136861cf8d766f1a56b"
        )

        with package.open("rb") as f:
            actual = hashlib.file_digest(f, "sha512").hexdigest()

        if actual != expected:
            raise RuntimeError("Download verification failed")

        with tarfile.open(package) as bundle:
            bundle.extractall("/content", filter="data")

        print("Kafka files verified and extracted.", flush=True)

    config = kafka_state / "server.properties"
    settings = (kafka_dir / "config/server.properties").read_text()

    overrides = {
        "listeners":
            "PLAINTEXT://127.0.0.1:9092,CONTROLLER://127.0.0.1:9093",
        "advertised.listeners":
            "PLAINTEXT://127.0.0.1:9092,CONTROLLER://127.0.0.1:9093",
        "controller.quorum.bootstrap.servers":
            "127.0.0.1:9093",
        "log.dirs":
            str(kafka_state / "data"),
    }

    for key, value in overrides.items():
        settings = re.sub(
            rf"(?m)^{re.escape(key)}=.*$",
            f"{key}={value}",
            settings
        )

    config.write_text(settings)

    storage = str(kafka_dir / "bin/kafka-storage.sh")

    if not (kafka_state / "data/meta.properties").exists():
        cluster_id = subprocess.check_output(
            [storage, "random-uuid"],
            text=True
        ).strip()

        subprocess.run(
            [
                storage, "format", "--standalone",
                "-t", cluster_id,
                "-c", str(config)
            ],
            check=True
        )

    with kafka_log.open("a") as log:
        kafka_process = subprocess.Popen(
            [
                str(kafka_dir / "bin/kafka-server-start.sh"),
                str(config)
            ],
            stdout=log,
            stderr=subprocess.STDOUT,
            env={
                **os.environ,
                "KAFKA_HEAP_OPTS": "-Xms256m -Xmx512m"
            },
            start_new_session=True
        )

print("Waiting for Kafka...", flush=True)

deadline = time.monotonic() + 60
ready = False

while time.monotonic() < deadline:
    admin = None

    try:
        admin = KafkaAdminClient(
            bootstrap_servers="127.0.0.1:9092",
            request_timeout_ms=5000,
            api_version_auto_timeout_ms=2000
        )
        admin.list_topics()
        ready = True
        break

    except Exception:
        time.sleep(2)

    finally:
        if admin is not None:
            admin.close()

if not ready:
    if kafka_log.exists():
        print(kafka_log.read_text(errors="replace")[-3000:])
    raise RuntimeError("Kafka did not become ready")

print("KAFKA_READY")

Waiting for Kafka...
KAFKA_READY


In [49]:
import os
import sys
import json
import subprocess
import uuid

from masar.native_contracts import validate_stage_result

lab_env = os.environ.copy()
for name in ("PYSPARK_GATEWAY_PORT", "PYSPARK_GATEWAY_SECRET"):
    lab_env.pop(name, None)

log_path = WORK / "reports" / f"lab05_console_{uuid.uuid4().hex}.log"
log_path.parent.mkdir(parents=True, exist_ok=True)

print("Starting Lab 05 with a fresh Spark process...", flush=True)
print("Downloading the connector and running the lab may take a few minutes.",
      flush=True)

try:
    with log_path.open("w") as log:
        execution = subprocess.run(
            [
                sys.executable, "-u",
                str(ROOT / "scripts/run_day04.py"),
                "--part", "streaming"
            ],
            cwd=str(ROOT),
            env=lab_env,
            stdout=log,
            stderr=subprocess.STDOUT,
            timeout=600
        )
except subprocess.TimeoutExpired:
    print(log_path.read_text(errors="replace")[-6000:])
    raise

if execution.returncode != 0:
    print(log_path.read_text(errors="replace")[-6000:])
    raise RuntimeError(f"Lab 05 failed. Log: {log_path}")

report_path = WORK / "reports/day04_stream_latest.json"
result = json.loads(report_path.read_text())
validate_stage_result("lab05_streaming", result)

print(json.dumps({
    "scope": result["scope"],
    "checks": result["checks"]
}, indent=2))

print("Transport rows:",
      [phase["transport_rows"] for phase in result["phases"]])
print("Unique event IDs:",
      [phase["unique_event_ids"] for phase in result["phases"]])
print("Evidence:", report_path)
print("LAB05_PASSED")

Starting Lab 05 with a fresh Spark process...
{
  "scope": "DAY04_NATIVE_STREAMING",
  "checks": {
    "actual_checkpoint_files_present": true,
    "all_events_link_to_trusted_trips": true,
    "checkpoint_same_for_all_phases": true,
    "event_content_matches_source": true,
    "late_event_retained": true,
    "phase_event_counts": true,
    "phase_transport_counts": true,
    "producer_consumer_offsets_reconcile": true,
    "restart_new_execution_ids": true,
    "restart_same_query_identity": true,
    "source_json_text_preserved": true,
    "transport_keys_always_unique": true,
    "unique_events_delta_readback": true
  }
}
Transport rows: [216, 216, 218, 219]
Unique event IDs: [216, 216, 216, 217]
Evidence: /content/masar-modern-data-engineering/outputs/day01_bronze_xgnqdyt3/reports/day04_stream_latest.json
LAB05_PASSED


Validate and quarantine

In [52]:
import os
import sys
import json
import subprocess
import uuid

print("Installing Lab 06 requirements...", flush=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "great-expectations==1.7.0",
        "pandas==2.2.3"
    ],
    check=True
)

from masar.native_contracts import validate_stage_result

lab_env = os.environ.copy()
for name in ("PYSPARK_GATEWAY_PORT", "PYSPARK_GATEWAY_SECRET"):
    lab_env.pop(name, None)

log_path = WORK / "reports" / f"lab06_console_{uuid.uuid4().hex}.log"
log_path.parent.mkdir(parents=True, exist_ok=True)

print("Running Lab 06...", flush=True)

try:
    with log_path.open("w") as log:
        execution = subprocess.run(
            [
                sys.executable, "-u",
                str(ROOT / "scripts/run_day04.py"),
                "--part", "quality"
            ],
            cwd=str(ROOT),
            env=lab_env,
            stdout=log,
            stderr=subprocess.STDOUT,
            timeout=600
        )
except subprocess.TimeoutExpired:
    print(log_path.read_text(errors="replace")[-6000:])
    raise

if execution.returncode != 0:
    print(log_path.read_text(errors="replace")[-6000:])
    raise RuntimeError(f"Lab 06 failed. Log: {log_path}")

report_path = WORK / "reports/day04_quality_latest.json"
result = json.loads(report_path.read_text())
validate_stage_result("lab06_quality", result)

print(json.dumps({
    "scope": result["scope"],
    "checks": result["checks"]
}, indent=2))

policy_path = WORK / result["reports"] / "mixed_policy.json"
policy = json.loads(policy_path.read_text())

print("Quarantined rows:", policy["rejected_rows"])

for phase in result["gx_results"]:
    print(
        phase["phase"],
        "rows:", phase["rows"],
        "success:", phase["success"]
    )

print("Evidence:", report_path)
print("LAB06_PASSED")

Installing Lab 06 requirements...
Running Lab 06...
{
  "scope": "DAY04_NATIVE_QUALITY",
  "checks": {
    "approved_readback_same_business_contents": true,
    "data_docs_exist_for_all_three_cases": true,
    "failed_candidate_not_promoted": true,
    "mixed_candidate_fails_gx": true,
    "native_mixed_counts": true,
    "native_reasons_match_reference": true,
    "quarantine_delta_readback": true,
    "source_silver_untouched": true,
    "trusted_and_rechecked_pass_gx": true
  }
}
Quarantined rows: 7
trusted rows: 75 success: True
mixed rows: 82 success: False
rechecked rows: 75 success: True
Evidence: /content/masar-modern-data-engineering/outputs/day01_bronze_xgnqdyt3/reports/day04_quality_latest.json
LAB06_PASSED


Receive Kafka events

In [54]:
import json
import zipfile
from google.colab import files
from masar.native_contracts import validate_stage_result

for stage, filename in [
    ("lab05_streaming", "day04_stream_latest.json"),
    ("lab06_quality", "day04_quality_latest.json")
]:
    report = json.loads((WORK / "reports" / filename).read_text())
    validate_stage_result(stage, report)
    print("Saved report verified:", stage)

archive = ROOT / "outputs/day04_handoff.zip"
pointer = ROOT / "outputs/day01_bronze_success.json"

with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())

with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None

files.download(str(archive))

Saved report verified: lab05_streaming
Saved report verified: lab06_quality


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Review and save